# CNN for Image Classification (multi-class classification)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

**torchvision** -> its is a official Pytorch library spacilly for computer vision
- its have **datasets** - CIFAR10, MNIST
- its also have lot of - **pretrained CNNS**
- and some importent tools or **utilities** for image transform  

## DAtasets & DataLoader

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # its help to perform transformation on images

# image => scale (0, 1) => normalize => (-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(), # auto convert all images to pytorch tensor and auto scale all images
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # in this params  define - what std-dev and mean values do we need after normalizing the image? 
])

# training set and testing set are automaticly available in CIFAR10, only we get
trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

Files already downloaded and verified


C:\Users\ranja\anaconda3\envs\ml\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified


**transforms.`compose`**: 
- its help to chain the lot of image transformation
- we don't apply on the time
- only we set all transformation earlier for the apply every image.
- its help us to do multi-transformation

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build CNN
- we get (32, 32, 3) -> (32, 32, 32)

- we use **3 convolutional layers** - mixed of convo + ReLU and maxpool
- convolutional layer is 2D beacuse we deal with images

- CNN => **[convolution+ReLU -> maxpool]**-3time => **flattening** => **FCL**
- convol - {kernel = 3 x 3 , padding = 1, stride = 1 (default)}
- maxpooling - {kernel = 2 x 2 , stride = 2}

- In **FCL** -> 1 HL (Linear + ReLU Activation func.)

-  than: **Final Output LAyer** - Linear Activation Func. (auto softmax beacuse of CrossEntropyLoss)

In [17]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # 1st layer of CNN
        self.conv_layers = nn.Sequential(
            # 1st convolution layer
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # input channel =3, output channel(feature maps)=32
            nn.ReLU(), # convo+ReLU -> (32, 32, 3) => (32, 32, 32)
            nn.MaxPool2d(2, 2), # kernel_size=2, stride=2
            # maxpool -> (32, 32, 32) => (16, 16, 32)
                
            # 2nd convolution layer
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # convo+ReLU -> (16, 16, 32) => (16, 16, 64)
            nn.MaxPool2d(2, 2), # maxpool -> (16, 16, 64) => (8, 8, 64)
            
            # 3rd convolution layer
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            # convo+ReLU -> (8, 8, 64) => (8, 8, 128)
            nn.MaxPool2d(2, 2) # maxpool -> (8, 8, 128) => (4, 4, 128)
            # after flatten the (4, 4, 128) => 4*48*128 inputs
        )

        # 2nd layer of CNN
        self.fc_layers  = nn.Sequential(
            nn.Linear(4*4*128, 256), # input, output=random
            nn.ReLU(),

            nn.Linear(256, 10) # o/p=10 because of multi-class classification
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening | by standord method
        x = self.fc_layers(x)

        return x

**conv2D()** -> create a 2D convolution layer
- why 2D?-> beacuse images are matrix form which is 2D (32 x 32)

### what is Output after 1st Conv. layer
- **Input**:
    - provided **Input** = (32, 32, 3) | image_size(WxH)=32x32 and **channel=3** (represent - RGB).
    - input metrics(32x32); **`n=32`**
    - kernel_size(filter)=3; **`f=3`**
    - Stride value = 1 (default); **`s=1`**
    - padding value = 1; **`p=1`**
- **Output**:
    - Formula => (n-f+2p)/s +1
    - after calculation output metrics is 32x32; **`output = 32`**;

### 2nd Parameter of `nn.conv2d()`
- define how many **Output channel** we wants
- this is value for total no. of feature maps
- generally it is power of 2.
- and it increases after evry layer.
- so 1st layer give - **output channel=32**
- and it also decide the how many filters which we wants to apply after applying the kernel in next time

### Output size after applying - `Convol + ReLU`
- (32, 32, 3) => **(32, 32, 32)** image size
- we pass 3 channel in Input and we want 32 channels in output
- beacuse we apply 32 filtters

### `maxPool2d()`
- when we apply **maxPool2d(2, 2)** here kernel_size=2, stride=2
- so maxpool2d(2, 2) - half the wheight and height
- (32, 32, 32) => (16, 16, 32)

### similarly:
- **2nd layer:**
    - convo+ReLU -> (16, 16, 32) => **(16, 16, 64)**  | output channel increase
    - maxpool -> (16, 16, 64) => **(8, 8, 64)**
- **3rd layer**
    - convo+ReLU -> (8, 8, 64) => **(8, 8, 128)** | output channel increase
    - maxpool -> (8, 8, 128) => **(4, 4, 128)**

### final Output image
- final **produced images** after all 3 convolution layers is => **(4, 4, 128)**

### Total no. of induvisual pixels
- means - how many inputs after do (4, 4, 128) => **flatten**
- So; `4*4*128` = ans.

In [18]:
model = CNN()

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [21]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images) #FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3596947458394044
epoch=2/10 & loss=0.9457060775869642
epoch=3/10 & loss=0.7566152687191658
epoch=4/10 & loss=0.6256928767846979
epoch=5/10 & loss=0.5188734935182134
epoch=6/10 & loss=0.421559967176841
epoch=7/10 & loss=0.3382689411587575
epoch=8/10 & loss=0.25517072352340153
epoch=9/10 & loss=0.19975939736037
epoch=10/10 & loss=0.1540162164879882


## Evaluatate our CNN

In [23]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad(): # no gradient compute because wo not do BP
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}")

Accuracy = 75.71
